# 5. Production Gotchas: State Management, Operations, and Evolution

- While the DataStream API defines how data moves, this section defines where that data lives during a crash and how to change your code without losing months of accumulated "memory.

## State Backend

- Flink offers two primary strategies for storing Keyed State. Your choice depends on the scale of your data and your latency requirements.
    - HashMapStateBackend
        - This stores state inside Java Heap (RAM)
        - It provides fast access, because there is no serialization overhead
        - BUT it is limited by TaskManager RAM; if the task manager runs out of memory, you risk losing data unless checkpointed

    - EmbeddedRocksDBStateBackend
        - This stores state on local disk 
        - It is slow to access, because you need to read from risk
        - BUT it supports huge amounts of state, potentially up to Terabytes!

    - For any production job where the state grows, use RocksDB for resilience to memory pressures

## Evolving your Stream: Operator UIDs

- In Flink, every stateful operator is assigned an ID. If you don't provide one, Flink generates it based on the DAG structure

- Let's suppose you decide that you add a new `map()` in your pipeline. Now the generated UIDs for all downstream tasks completely change!

- When you try to restart from a Savepoint, Flink won't find the state for the "new" IDs, and your job will start from zero. 
    - If your metric was accumulating over, say, 1 hour, you get 1 hour of no data before your app outputs!

- **SOLUTION:** Always manually assign UIDs to every stateful operator!

    ```java
    // DO THIS for every stateful operator!
    DataStream<Alert> alerts = stream
        .keyBy(log -> log.getUserId())
        .window(TumblingEventTimeWindows.of(Time.hours(1)))
        .aggregate(new MyAggregator())
        .uid("user-purchase-agg-001"); // This ID must NEVER change
    ```

## State TTL (Time-To-Live)

- Flink stateful operators accumulate state, and Flink NEVER invalidates old states, even if it is not being used any more!

- So if you checkpoint your app, and you have some state that was previously stored, that lives in your checkpoints **FOREVER**

- It is always best practise to use State TTL to automatically expire old keys

    ```java
    StateTtlConfig ttlConfig = StateTtlConfig
        .newBuilder(Time.days(7)) // Delete state if not touched for 7 days
        .setUpdateType(StateTtlConfig.UpdateType.OnCreateAndWrite)
        .setStateVisibility(StateTtlConfig.StateVisibility.NeverReturnExpired)
        .build();

    ValueStateDescriptor<Long> desc = new ValueStateDescriptor<>("lastSeen", Long.class);
    desc.enableTimeToLive(ttlConfig); // Enable cleanup!
    ```

### Scaling: Parallelism vs. Max Parallelism

- Scaling a Flink job is more restrictive than scaling a web app because state is partitioned into "Key Groups".
    - Parallelism: The current number of active Task Slots.
    - Max Parallelism: The number of "Key Groups" created **when the job first starts**. This defines the absolute ceiling for scaling.

- So even if you change Parallelism (e.g., 10 → 20) during a restart from a Savepoint, you cannot change Max Parallelism. 
    - If you set Max Parallelism to 128, you can never scale beyond 128 slots without throwing away your state and starting over!!

    

### Backpressure

- When a Flink job is slow, it is usually because one operator cannot keep up with the data, causing a "traffic jam" upstream.
- How to find it: Check the Flink Web UI. Operators under pressure will show a high Backpressure status (colored red).
    - Common Culprits: * An external database sink that is too slow.
    - Skewed data (e.g., one "Taylor Swift" key overwhelming a single TaskManager).
    - Complex logic in a ProcessFunction that needs more CPU.